# 05 Features

Adds the features the models use to every dataset. Nothing here looks at a departure's own label or block time.

Each dataset is processed on its own: counts and queues are built from the movements in that file. For ranking and
final ranking, the 2025 training movements are added as history, so the first hours of a month still see the traffic
before them. Features are filled for departures only; arrival rows keep missing values.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir() and (p / "pyproject.toml").is_file())
CLEAN = ROOT / "data/02_clean"
OUT = ROOT / "data/03_features"
EXTRA = ROOT / "data/04_additional_data"
ADSB_FEATURES = ROOT / "data/03_features/adsb_movement_movement_1s_v5.parquet"   # written by 04_adsb_movement

TRAINING = {"departures": "train_DEP_merged_sorted.parquet", "arrivals": "train_ARR_merged_sorted.parquet"}
OTHER = ["ranking.parquet", "final_ranking.parquet"]   # files with both phases; missing files are skipped
OUT.mkdir(parents=True, exist_ok=True)

## Load

`datasets` holds the departures to describe (one frame per output file); `movements` holds every movement that may
serve as context for that file.

In [2]:
COLUMNS = ["MVT_ID_mvt", "FLIGHT_ID_mvt", "PHASE_mvt", "ADEP_mvt", "ADES_mvt", "RUNWAY_mvt", "STAND_mvt",
           "MVT_TIME_UTC_mvt", "SCHED_TIME_UTC_mvt", "BLOCK_TIME_UTC_mvt", "TAXITIME_SEC_mvt", "AIRCRAFT_TYPE_mvt",
           "AOBT_3_flt", "EOBT_1_flt", "LOBT_flt", "IOBT_flt", "ARVT_1_flt", "ARVT_3_flt", "FLIGHT_mvt"]


def read(name):
    """Departure labels are blanked on load, so no feature can touch them by mistake."""
    frame = pd.read_parquet(CLEAN / name, columns=COLUMNS)
    frame.loc[frame["PHASE_mvt"].eq("DEP"), ["BLOCK_TIME_UTC_mvt", "TAXITIME_SEC_mvt"]] = np.nan
    return frame


training_movements = pd.concat([read(TRAINING["departures"]), read(TRAINING["arrivals"])], ignore_index=True)
movements = {TRAINING["departures"]: training_movements}
for name in OTHER:
    if (CLEAN / name).is_file():
        movements[name] = pd.concat([training_movements, read(name)], ignore_index=True)
own_rows = {name: read(name) for name in movements}   # rows of the output file itself (DEP and ARR for ranking files)
pd.DataFrame({name: {"rows": len(df), "departures": int(df.PHASE_mvt.eq("DEP").sum()),
                     "context movements": len(movements[name])} for name, df in own_rows.items()}).T

,rows,departures,context movements
train_DEP_merged_sorted.parquet,2085047,2085047,4167797
ranking.parquet,689534,344841,4857331
final_ranking.parquet,1341197,670790,5508994


In [3]:
def seconds(values):
    """UTC timestamps as float seconds since 1970; missing stays NaN."""
    ts = pd.to_datetime(values, utc=True)
    return np.where(ts.isna(), np.nan, ts.dt.as_unit("ns").astype("int64") / 1e9)


def ns(values):
    return pd.to_datetime(values, utc=True).dt.as_unit("ns").astype("int64").to_numpy()


def local_airport(frame):
    """Departures happen at ADEP, arrivals at ADES."""
    return frame["ADEP_mvt"].where(frame["PHASE_mvt"].eq("DEP"), frame["ADES_mvt"]).where(frame["PHASE_mvt"].isin(["DEP", "ARR"]))


features = {name: pd.DataFrame(index=df.index[df.PHASE_mvt.eq("DEP")]) for name, df in own_rows.items()}
departures = {name: df.loc[df.PHASE_mvt.eq("DEP")] for name, df in own_rows.items()}

## NM and schedule times

Differences between the take-off time, the schedule and the NM times (all in seconds).

In [4]:
PAIRS = {   # feature: (later time, earlier time)
    "mvt_minus_aobt_sec": ("MVT_TIME_UTC_mvt", "AOBT_3_flt"),
    "movement_minus_scheduled_offblock_sec": ("MVT_TIME_UTC_mvt", "SCHED_TIME_UTC_mvt"),
    "delay_sec": ("MVT_TIME_UTC_mvt", "SCHED_TIME_UTC_mvt"),   # same value; the name used by classifier and corrector
    "mvt_minus_lobt_sec": ("MVT_TIME_UTC_mvt", "LOBT_flt"),
    "mvt_minus_eobt_sec": ("MVT_TIME_UTC_mvt", "EOBT_1_flt"),
    "mvt_minus_iobt_sec": ("MVT_TIME_UTC_mvt", "IOBT_flt"),
    "mvt_minus_arvt1_sec": ("MVT_TIME_UTC_mvt", "ARVT_1_flt"),
    "mvt_minus_arvt3_sec": ("MVT_TIME_UTC_mvt", "ARVT_3_flt"),
    "sched_minus_eobt_sec": ("SCHED_TIME_UTC_mvt", "EOBT_1_flt"),
    "sched_minus_iobt_sec": ("SCHED_TIME_UTC_mvt", "IOBT_flt"),
    "aobt_minus_sched_sec": ("AOBT_3_flt", "SCHED_TIME_UTC_mvt"),
    "aobt_minus_eobt_sec": ("AOBT_3_flt", "EOBT_1_flt"),
    "lobt_minus_eobt_sec": ("LOBT_flt", "EOBT_1_flt"),
}
for name, dep in departures.items():
    t = lambda column: pd.to_datetime(dep[column], utc=True)
    for feature, (later, earlier) in PAIRS.items():
        features[name][feature] = (t(later) - t(earlier)).dt.total_seconds()
    features[name]["aobt_missing"] = dep["AOBT_3_flt"].isna().astype(float)
    features[name]["nm_missing"] = dep["LOBT_flt"].isna().astype(float)
    features[name]["aobt_has_seconds"] = np.where(t("AOBT_3_flt").isna(), np.nan, (t("AOBT_3_flt").dt.second != 0).astype(float))
    features[name]["mvt_hour"] = t("MVT_TIME_UTC_mvt").dt.hour.astype(float)   # UTC hour
    # airline prefix of the flight number, e.g. "DLH" from "DLH4AB"
    features[name]["carrier"] = dep["FLIGHT_mvt"].astype("string").str.extract(r"^([A-Z0-9]{2}[A-Z]?)\d", expand=False)
features[TRAINING["departures"]].describe().T.round(1)

,count,mean,std,min,25%,50%,75%,max
mvt_minus_aobt_sec,2062577.0,1003.9,377.2,-11521.0,734.0,958.0,1201.0,87181.0
movement_minus_scheduled_offblock_sec,2085047.0,1892.5,2300.7,-113526.0,917.0,1402.0,2209.0,270508.0
delay_sec,2085047.0,1892.5,2300.7,-113526.0,917.0,1402.0,2209.0,270508.0
mvt_minus_lobt_sec,2062577.0,1352.8,749.1,-3382.0,850.0,1237.0,1705.0,87001.0
mvt_minus_eobt_sec,2062577.0,1364.1,674.1,-9390.0,905.0,1255.0,1679.0,87001.0
mvt_minus_iobt_sec,2062577.0,1353.8,748.8,-3382.0,853.0,1238.0,1704.0,87001.0
mvt_minus_arvt1_sec,2062577.0,-10317.9,10861.9,-62035.0,-10801.0,-5813.0,-3522.0,83542.0
mvt_minus_arvt3_sec,2062577.0,-10676.1,10921.4,-115120.0,-11198.0,-6154.0,-3741.0,84061.0
sched_minus_eobt_sec,2062577.0,-490.3,2075.6,-268800.0,-420.0,0.0,0.0,114300.0
sched_minus_iobt_sec,2062577.0,-500.6,2046.8,-268800.0,0.0,0.0,0.0,114300.0


## Local time

Weekday as a cycle and the hour, both in the airport's local time (summer time included).

In [5]:
AIRPORT_TIMEZONES = {"EDDF": "Europe/Berlin", "EDDM": "Europe/Berlin", "EGLL": "Europe/London",
                     "EHAM": "Europe/Amsterdam", "LEBL": "Europe/Madrid", "LEMD": "Europe/Madrid",
                     "LFPG": "Europe/Paris", "LIRF": "Europe/Rome", "LSZH": "Europe/Zurich", "LTFM": "Europe/Istanbul"}

for name, dep in departures.items():
    mvt = pd.to_datetime(dep["MVT_TIME_UTC_mvt"], utc=True)
    weekday, hour = pd.Series(np.nan, index=dep.index), pd.Series(np.nan, index=dep.index)
    for airport, zone in AIRPORT_TIMEZONES.items():
        rows = dep["ADEP_mvt"].eq(airport)
        local = mvt[rows].dt.tz_convert(zone)
        weekday[rows], hour[rows] = local.dt.dayofweek, local.dt.hour
    unknown = set(dep["ADEP_mvt"].dropna()) - set(AIRPORT_TIMEZONES)
    assert not unknown, f"No time zone for {unknown}"
    features[name]["movement_day_of_week_sin"] = np.sin(2 * np.pi * weekday / 7)
    features[name]["movement_day_of_week_cos"] = np.cos(2 * np.pi * weekday / 7)
    features[name]["local_hour"] = hour

## Traffic around the take-off

Departures in the 30 minutes before the take-off (airport and own runway), departures on the own runway in the
30 minutes after, and the time since the last movement on the runway. Intervals are open, so the flight never counts
itself.

In [6]:
def sorted_times(frame, key, phase=None):
    frame = frame if phase is None else frame.loc[frame["PHASE_mvt"].eq(phase)]
    keys = key(frame)
    times = pd.Series(ns(frame["MVT_TIME_UTC_mvt"]), index=frame.index)
    return {k: np.sort(v.to_numpy()) for k, v in times.groupby(keys) if pd.notna(k)}


def airport_key(frame):
    return local_airport(frame)


def runway_key(frame):
    return local_airport(frame).str.cat(frame["RUNWAY_mvt"], sep="|")


def window_counts(dep, key, lookup, window_ns=pd.Timedelta(minutes=30).value):
    query = ns(dep["MVT_TIME_UTC_mvt"])
    keys = key(dep).to_numpy()
    before, after = np.full(len(dep), np.nan), np.full(len(dep), np.nan)
    before[pd.notna(keys)] = after[pd.notna(keys)] = 0
    for k, times in lookup.items():
        rows = keys == k
        q = query[rows]
        before[rows] = np.searchsorted(times, q, "left") - np.searchsorted(times, q - window_ns, "right")
        after[rows] = np.searchsorted(times, q + window_ns, "left") - np.searchsorted(times, q, "right")
    return before, after


for name, dep in departures.items():
    context = movements[name]
    before, _ = window_counts(dep, airport_key, sorted_times(context, airport_key, "DEP"))
    features[name]["deps_before_30min"] = before
    before, after = window_counts(dep, runway_key, sorted_times(context, runway_key, "DEP"))
    features[name]["deps_before_30min_same_runway"] = before
    features[name]["deps_after_30min_same_runway"] = after

In [7]:
for name, dep in departures.items():
    lookup = sorted_times(movements[name], runway_key)
    query, keys = ns(dep["MVT_TIME_UTC_mvt"]), runway_key(dep).to_numpy()
    elapsed = np.full(len(dep), np.nan)
    for k, times in lookup.items():
        rows = keys == k
        position = np.searchsorted(times, query[rows], "left") - 1
        elapsed[rows] = np.where(position >= 0, (query[rows] - times[np.clip(position, 0, None)]) / 1e9, np.nan)
    features[name]["seconds_since_last_movement"] = elapsed

## Departure flow at the airport

Built from the departures of the same file. A delay wave, a pause in departures or a drop in the departure rate are
signs of disruption. The typical rate per airport and UTC hour comes from 2025 and is reused for the other files.

In [8]:
def departure_flow(dep, typical_rate=None):
    mvt = seconds(dep["MVT_TIME_UTC_mvt"])
    delay = mvt - seconds(dep["SCHED_TIME_UTC_mvt"])
    runway = dep["RUNWAY_mvt"].astype("string").fillna("?").to_numpy(dtype=object)
    utc_hour = pd.to_datetime(dep["MVT_TIME_UTC_mvt"], utc=True).dt.hour.to_numpy()
    n = len(dep)
    med30, gap2h, rate60, share30, nrw60, q10 = (np.full(n, np.nan) for _ in range(6))
    airports = dep["ADEP_mvt"].to_numpy()
    for airport in pd.unique(airports):
        p = np.flatnonzero(airports == airport)
        o = p[np.argsort(mvt[p], kind="stable")]
        t, dl, rw = mvt[o], delay[o], runway[o]
        lo30, hi30 = np.searchsorted(t, t - 1800, "left"), np.searchsorted(t, t + 1800, "right")
        lo60 = np.searchsorted(t, t - 3600, "left")
        lo10, hi10 = np.searchsorted(t, t - 600, "left"), np.searchsorted(t, t + 600, "right")
        m30, s30, k60, q = (np.full(len(t), np.nan) for _ in range(4))
        for i in range(len(t)):
            window = dl[lo30[i]:hi30[i]]
            if len(window) > 1:
                m30[i] = np.median(np.delete(window, i - lo30[i]))   # the other departures only
            before30 = rw[lo30[i]:i]
            if len(before30):
                s30[i] = np.mean(before30 == rw[i])
            k60[i] = len(set(rw[lo60[i]:i]))
            q[i] = np.count_nonzero(rw[lo10[i]:hi10[i]] == rw[i]) - 1
        gaps = pd.Series(np.diff(t, prepend=np.nan), index=pd.to_datetime(t, unit="s")).rolling("2h").max().to_numpy()
        med30[o], gap2h[o], share30[o], nrw60[o], q10[o] = m30, gaps, s30, k60, q
        rate60[o] = np.arange(len(t)) - lo60
    key = pd.Series(airports.astype(str)) + "|" + pd.Series(utc_hour.astype(str))
    if typical_rate is None:
        typical_rate = pd.Series(rate60).groupby(key.to_numpy()).median()
    typical = key.map(typical_rate).to_numpy(dtype=float)
    ratio = np.where(typical > 0, rate60 / np.where(typical > 0, typical, 1), np.nan)
    flow = pd.DataFrame({"dep_median_delay_30min_sec": med30, "dep_max_gap_2h_sec": gap2h, "dep_rate_60min_ratio": ratio,
                         "dep_runway_share_30min": share30, "dep_n_runways_60min": nrw60,
                         "deps_same_runway_10min": q10}, index=dep.index)
    return flow, typical_rate


typical_rate = None
for name, dep in departures.items():   # training comes first and sets the typical rate
    flow, typical_rate = departure_flow(dep, typical_rate)
    for column in flow:
        features[name][column] = flow[column]
features[TRAINING["departures"]].filter(like="dep_").describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
dep_median_delay_30min_sec,2082917.0,1530.57,723.69,-35136.0,1111.00,1394.00,1771.00,63783.0
dep_max_gap_2h_sec,2085037.0,1524.94,4320.74,66.0,319.00,480.00,727.00,79254.0
dep_rate_60min_ratio,2084249.0,0.99,0.27,0.0,0.86,1.00,1.12,19.0
dep_runway_share_30min,2077253.0,0.69,0.27,0.0,0.48,0.67,1.00,1.0
dep_n_runways_60min,2085047.0,1.87,0.63,0.0,1.00,2.00,2.00,7.0


## Queue in front of the departure

The off-block reference is AOBT, or EOBT when AOBT is missing. Counts how many aircraft were taxiing, waiting at the
gate past their EOBT or still rolling in after landing at that moment, and how long the other departures of the last
30/60 minutes needed from off-block to take-off. Uses only the NM times of the other departures, never their labels.

In [9]:
NEIGHBOUR_MAX_SEC = 7200   # plausible MVT - AOBT values only


def window_median(times, values, query, width):
    order = np.argsort(times, kind="stable")
    t, v = times[order], values[order]
    lo, hi = np.searchsorted(t, query - width, "left"), np.searchsorted(t, query, "left")
    return np.array([np.median(v[a:b]) if b > a else np.nan for a, b in zip(lo, hi)])


def open_intervals(starts, ends, query):
    """How many intervals [start, end) contain each query time."""
    ok = np.isfinite(starts) & np.isfinite(ends) & (ends > starts)
    s, e = np.sort(starts[ok]), np.sort(ends[ok])
    out = (np.searchsorted(s, query, "right") - np.searchsorted(e, query, "right")).astype(float)
    return np.where(np.isfinite(query), out, np.nan)


def runway_config(times, runways, query, width=3600):
    order = np.argsort(times, kind="stable")
    t, r = times[order], runways[order]
    lo, hi = np.searchsorted(t, query - width, "left"), np.searchsorted(t, query, "left")
    return np.array(["|".join(sorted(set(r[a:b]))) if b > a else None for a, b in zip(lo, hi)], dtype=object)

In [10]:
def queue_features(dep, arr):
    mvt, aobt, eobt = seconds(dep["MVT_TIME_UTC_mvt"]), seconds(dep["AOBT_3_flt"]), seconds(dep["EOBT_1_flt"])
    offblock = np.where(np.isfinite(aobt), aobt, eobt)
    taxi = mvt - aobt
    taxi = np.where((taxi >= 0) & (taxi <= NEIGHBOUR_MAX_SEC), taxi, np.nan)
    runway = dep["RUNWAY_mvt"].astype("string").fillna("?").to_numpy(dtype=object)
    airports = dep["ADEP_mvt"].to_numpy()
    a_mvt, a_block = seconds(arr["MVT_TIME_UTC_mvt"]), seconds(arr["BLOCK_TIME_UTC_mvt"])
    a_airport = arr["ADES_mvt"].to_numpy()
    a_runway = arr["RUNWAY_mvt"].astype("string").fillna("?").to_numpy(dtype=object)
    n = len(dep)
    out = {c: np.full(n, np.nan) for c in ["nbr_mvt_minus_aobt_30min_sec", "nbr_mvt_minus_aobt_60min_sec",
           "nbr_mvt_minus_aobt_rwy_60min_sec", "deps_between_offblock_and_mvt", "deps_same_runway_between_offblock_and_mvt",
           "taxi_queue_at_offblock", "gate_retention_at_offblock", "arr_taxiing_at_offblock"]}
    out["dep_runway_config_60min"] = np.full(n, None, dtype=object)
    out["arr_runway_config_60min"] = np.full(n, None, dtype=object)
    for airport in pd.unique(airports):
        p = np.flatnonzero(airports == airport)
        valid = p[np.isfinite(taxi[p])]
        out["nbr_mvt_minus_aobt_30min_sec"][p] = window_median(mvt[valid], taxi[valid], mvt[p], 1800)
        out["nbr_mvt_minus_aobt_60min_sec"][p] = window_median(mvt[valid], taxi[valid], mvt[p], 3600)
        for rw in pd.unique(runway[p]):
            pr = p[runway[p] == rw]
            vr = pr[np.isfinite(taxi[pr])]
            out["nbr_mvt_minus_aobt_rwy_60min_sec"][pr] = window_median(mvt[vr], taxi[vr], mvt[pr], 3600)
            t_rw = np.sort(mvt[pr])
            between = np.searchsorted(t_rw, mvt[pr], "left") - np.searchsorted(t_rw, offblock[pr], "right")
            out["deps_same_runway_between_offblock_and_mvt"][pr] = np.where(np.isfinite(offblock[pr]), np.maximum(between, 0), np.nan)
        t_all = np.sort(mvt[p])
        between = np.searchsorted(t_all, mvt[p], "left") - np.searchsorted(t_all, offblock[p], "right")
        out["deps_between_offblock_and_mvt"][p] = np.where(np.isfinite(offblock[p]), np.maximum(between, 0), np.nan)
        own = np.isfinite(aobt[p]) & (mvt[p] > aobt[p])
        out["taxi_queue_at_offblock"][p] = open_intervals(aobt[p], mvt[p], offblock[p]) - own
        out["gate_retention_at_offblock"][p] = open_intervals(eobt[p], aobt[p], offblock[p])
        a = np.flatnonzero(a_airport == airport)
        out["arr_taxiing_at_offblock"][p] = open_intervals(a_mvt[a], a_block[a], offblock[p])
        out["dep_runway_config_60min"][p] = runway_config(mvt[p], runway[p], mvt[p])
        a_ok = a[np.isfinite(a_mvt[a])]
        out["arr_runway_config_60min"][p] = runway_config(a_mvt[a_ok], a_runway[a_ok], mvt[p])
    return pd.DataFrame(out, index=dep.index)


for name, dep in departures.items():
    file_rows = own_rows[name] if name != TRAINING["departures"] else read(TRAINING["arrivals"])
    queue = queue_features(dep, file_rows.loc[file_rows.PHASE_mvt.eq("ARR")])
    for column in queue:
        features[name][column] = queue[column].astype("string") if column.endswith("config_60min") else queue[column]

## Stand and inbound aircraft

Arrival block times are public, so they show what happened at the departure's stand: when the last aircraft parked
there, how many arrived between the scheduled off-block and the take-off, and when the likely own inbound aircraft (same
type, at least 15 min earlier) reached the stand. A stand that was re-used before the take-off hints at a longer taxi
time, and the arrival of the own inbound aircraft helps to recognise labels that equal the delay; neither is a strict
rule.

In [11]:
STAND_LOOKBACK_SEC, SCHED_MARGIN_SEC, INBOUND_MIN_GAP_SEC, ARR_WINDOW_SEC = 86400, 600, 900, 3600


def arrival_lookups(arrivals):
    arrivals = arrivals.loc[arrivals["BLOCK_TIME_UTC_mvt"].notna() & arrivals["ADES_mvt"].notna()].copy()
    arrivals["inblock"] = seconds(arrivals["BLOCK_TIME_UTC_mvt"])
    arrivals["stand_key"] = arrivals["ADES_mvt"].astype(str) + "|" + arrivals["STAND_mvt"].astype("string")
    stands, taxi_in = {}, {}
    for key, group in arrivals.dropna(subset=["STAND_mvt"]).groupby("stand_key"):
        order = np.argsort(group["inblock"].to_numpy(), kind="stable")
        stands[key] = (group["inblock"].to_numpy()[order],
                       group["AIRCRAFT_TYPE_mvt"].astype("string").fillna("").to_numpy(dtype=object)[order])
    for airport, group in arrivals.dropna(subset=["TAXITIME_SEC_mvt"]).groupby("ADES_mvt"):
        order = np.argsort(group["inblock"].to_numpy(), kind="stable")
        taxi = np.clip(group["TAXITIME_SEC_mvt"].to_numpy(dtype=float)[order], 0, 3600)
        taxi_in[airport] = (group["inblock"].to_numpy()[order], np.concatenate([[0.0], np.cumsum(taxi)]))
    return stands, taxi_in

In [12]:
def stand_features(dep, stands, taxi_in):
    columns = ["stand_reocc_gap_sec", "stand_reocc_other_type", "stand_first_arr_after_sched_gap_sec",
               "stand_n_arr_sched_to_mvt", "stand_prev_arr_gap_sec", "stand_inbound_minus_sched_sec", "arr_taxi_in_last_hour_sec"]
    n = len(dep)
    out = {c: np.full(n, np.nan) for c in columns}
    mvt, sched = seconds(dep["MVT_TIME_UTC_mvt"]), seconds(dep["SCHED_TIME_UTC_mvt"])
    dep_type = dep["AIRCRAFT_TYPE_mvt"].astype("string").fillna("").to_numpy(dtype=object)
    stand_key = (dep["ADEP_mvt"].astype(str) + "|" + dep["STAND_mvt"].astype("string")).fillna("").to_numpy(dtype=object)
    has_stand = dep["STAND_mvt"].notna().to_numpy()
    out["stand_n_arr_sched_to_mvt"][has_stand] = 0.0
    for key in pd.unique(stand_key[has_stand]):
        if key not in stands:
            continue
        times, types = stands[key]
        idx = np.flatnonzero(has_stand & (stand_key == key))
        q = mvt[idx]
        last = np.searchsorted(times, q, "left") - 1               # latest arrival before take-off
        ok = last >= 0
        gap = np.where(ok, q - times[np.clip(last, 0, None)], np.nan)
        ok &= gap <= STAND_LOOKBACK_SEC
        out["stand_reocc_gap_sec"][idx[ok]] = gap[ok]
        out["stand_reocc_other_type"][idx[ok]] = ((types[last[ok]] != dep_type[idx[ok]]) & (types[last[ok]] != "")).astype(float)
        lo = np.searchsorted(times, sched[idx] - SCHED_MARGIN_SEC, "right")
        hi = np.searchsorted(times, q, "left")
        count = np.maximum(hi - lo, 0)
        out["stand_n_arr_sched_to_mvt"][idx] = count
        has = count > 0
        out["stand_first_arr_after_sched_gap_sec"][idx[has]] = q[has] - times[lo[has]]
        prev = np.searchsorted(times, q - INBOUND_MIN_GAP_SEC, "right") - 1   # likely own inbound
        okp = prev >= 0
        prev_gap = np.where(okp, q - times[np.clip(prev, 0, None)], np.nan)
        okp &= prev_gap <= STAND_LOOKBACK_SEC
        out["stand_prev_arr_gap_sec"][idx[okp]] = prev_gap[okp]
        found = np.zeros(len(idx), dtype=bool)
        for back in range(5):                                       # same aircraft type, up to 5 arrivals back
            j = prev - back
            okj = (j >= 0) & ~found
            tj = times[np.clip(j, 0, None)]
            okj &= (q - tj) <= STAND_LOOKBACK_SEC
            same = okj & (types[np.clip(j, 0, None)] == dep_type[idx]) & (dep_type[idx] != "")
            out["stand_inbound_minus_sched_sec"][idx[same]] = tj[same] - sched[idx[same]]
            found |= same
    airports = dep["ADEP_mvt"].to_numpy()
    for airport in pd.unique(airports):
        if airport not in taxi_in:
            continue
        times, cumulative = taxi_in[airport]
        idx = np.flatnonzero(airports == airport)
        lo = np.searchsorted(times, mvt[idx] - ARR_WINDOW_SEC, "right")
        hi = np.searchsorted(times, mvt[idx], "right")
        count = hi - lo
        has = count > 0
        out["arr_taxi_in_last_hour_sec"][idx[has]] = (cumulative[hi[has]] - cumulative[lo[has]]) / count[has]
    return pd.DataFrame(out, index=dep.index)


for name, dep in departures.items():
    context = movements[name]
    stands, taxi_in = arrival_lookups(context.loc[context.PHASE_mvt.eq("ARR")])
    for column, values in stand_features(dep, stands, taxi_in).items():
        features[name][column] = values

## Weather

NOAA GHCNh hourly reports (CC0). Each departure gets the latest report of its airport at or before take-off, at most
two hours old. Zurich uses the reports of Dübendorf airfield, 10.5 km away, because GHCNh has no current station at
the airport. Only temperature and visibility are used by the models. If NOAA has several records for the same
time, the record with more aviation (FM-15/16) fields, then more filled fields, wins.

In [13]:
from urllib.request import Request, urlopen

NOAA = "https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly"
WEATHER_DIR = EXTRA / "raw_ghcnh"
WEATHER_DIR.mkdir(parents=True, exist_ok=True)
MAX_AGE = pd.Timedelta(hours=2)


def noaa_file(relative, name, optional=False):
    path = WEATHER_DIR / name
    if not path.exists():
        try:
            with urlopen(Request(f"{NOAA}/{relative}", headers={"User-Agent": "PRC-2026"}), timeout=120) as response:
                path.with_suffix(".part").write_bytes(response.read())
            path.with_suffix(".part").replace(path)
        except Exception as error:
            if optional and "404" in str(error):
                return None
            raise
    return path


stations = pd.read_csv(noaa_file("doc/ghcnh-station-list.csv", "ghcnh-station-list.csv"), dtype="string")
stations["ICAO"] = stations["ICAO"].str.strip()
airports = sorted(set().union(*[set(d["ADEP_mvt"].dropna()) for d in departures.values()]))
# Zurich has no GHCNh station with the ICAO code LSZH, and the Kloten station has no files after 2023. The METAR
# station of Dübendorf airfield (LSMD), 10.5 km away, stands in for it.
STAND_IN = {"LSZH": "SZI0000LSMD"}
station_of = {a: stations.loc[stations.ICAO.eq(a), "GHCN_ID"].item() for a in airports
              if stations.ICAO.eq(a).sum() == 1}
station_of.update({a: s for a, s in STAND_IN.items() if a in airports})
print("Airports without an exact ICAO station:", sorted(set(airports) - set(station_of)) or "none")
years = sorted(set().union(*[set(pd.to_datetime(d["MVT_TIME_UTC_mvt"], utc=True).dt.year) for d in departures.values()]))
years = sorted(set(years) | {min(years) - 1})   # reports from the evening before the first day

Airports without an exact ICAO station: none


In [14]:
# All weather fields are read for the duplicate tie-break, even though only two of them are kept.
FIELDS = {"temperature": "weather_temperature_c", "visibility": "weather_visibility_km", "dew_point_temperature": "dewpoint",
          "relative_humidity": "humidity", "wind_speed": "wind_speed", "wind_direction": "wind_direction", "wind_gust": "gust",
          "sea_level_pressure": "slp", "station_level_pressure": "stp", "altimeter": "altimeter", "precipitation": "precip",
          **{f"sky_cover_layer_baseht_{i}": f"cloud_base_{i}" for i in (1, 2, 3)},
          **{f"pres_wx_{k}{i}": f"present_{k}{i}" for k in ("MW", "AU", "AW") for i in (1, 2, 3)},
          **{f"sky_cover_layer_{i}": f"sky_{i}" for i in (1, 2, 3)},
          "precipitation_Measurement_Code": "precip_code", "wind_direction_Measurement_Code": "wind_dir_code"}


def read_station_year(airport, station, year):
    path = noaa_file(f"access/by-year/{year}/psv/GHCNh_{station}_{year}.psv", f"GHCNh_{station}_{year}.psv", optional=True)
    if path is None:
        return None
    header = pd.read_csv(path, sep="|", nrows=0).columns
    fields = dict(FIELDS)
    for i in (1, 2, 3):   # older header names
        for new, old in [(f"sky_cover_layer_{i}", f"sky_cover_{i}"), (f"sky_cover_layer_baseht_{i}", f"sky_cov_baseht_{i}")]:
            if new not in header and old in header:
                fields[old] = fields.pop(new)
    wanted = set(fields) | {"STATION", "DATE"} | {c for c in header if c.endswith("_Report_Type")}
    raw = pd.read_csv(path, sep="|", usecols=lambda c: c in wanted, dtype="string", keep_default_na=False,
                      na_values=["", "-9999", "-9999.0", "-9999.00"])
    out = pd.DataFrame({"ADEP_mvt": airport, "time": pd.to_datetime(raw["DATE"], utc=True).dt.as_unit("ns"),
                        "file": path.name, "row": np.arange(len(raw))})
    for source, target in fields.items():
        values = raw[source] if source in raw else pd.Series(pd.NA, index=raw.index, dtype="string")
        out[target] = values if target.startswith(("present", "sky_", "precip_code", "wind_dir_code")) else \
            pd.to_numeric(values, errors="raise").astype("float64").mask(lambda v: v.eq(-9999))
    no_bearing = out["wind_direction"].eq(999) & out["wind_dir_code"].isin(["V", "C"])
    out.loc[no_bearing, "wind_direction"] = np.nan
    out["dewpoint_spread"] = out["weather_temperature_c"] - out["dewpoint"]
    reports = [c for c in raw if c.endswith("_Report_Type")]
    out["aviation_fields"] = sum((raw[c].str.match(r"^FM1[56](?:$|[:\-])", na=False).astype(int) for c in reports),
                                 pd.Series(0, index=raw.index)).to_numpy()
    out["filled_fields"] = out[list(fields.values()) + ["dewpoint_spread"]].notna().sum(axis=1)
    return out


reports = pd.concat([r for a, s in station_of.items() for y in years if (r := read_station_year(a, s, y)) is not None],
                    ignore_index=True)
reports = reports.sort_values(["ADEP_mvt", "time", "aviation_fields", "filled_fields", "file", "row"],
                              ascending=[True, True, False, False, True, True], kind="stable")
reports = reports.drop_duplicates(["ADEP_mvt", "time"])[["ADEP_mvt", "time", "weather_temperature_c", "weather_visibility_km"]]
print(f"{len(reports):,} reports from {len(station_of)} stations, years {years}")

615,205 reports from 10 stations, years [2024, 2025, 2026]


In [15]:
for name, dep in departures.items():
    keys = pd.DataFrame({"ADEP_mvt": dep["ADEP_mvt"].to_numpy(), "position": np.arange(len(dep)),
                         "time": pd.to_datetime(dep["MVT_TIME_UTC_mvt"], utc=True).dt.as_unit("ns").to_numpy()})
    values = pd.DataFrame(np.nan, index=range(len(dep)), columns=["weather_temperature_c", "weather_visibility_km"])
    for airport, flights in keys.dropna().groupby("ADEP_mvt"):
        station = reports.loc[reports.ADEP_mvt.eq(airport)].drop(columns="ADEP_mvt")
        matched = pd.merge_asof(flights.sort_values("time"), station, on="time", direction="backward", tolerance=MAX_AGE)
        values.loc[matched["position"].to_numpy()] = matched[values.columns].to_numpy()
    for column in values:
        features[name][column] = values[column].to_numpy()
    print(name, "with weather:", f"{features[name]['weather_temperature_c'].notna().mean():.1%}")

train_DEP_merged_sorted.parquet with weather: 98.6%


ranking.parquet with weather: 99.8%


final_ranking.parquet with weather: 99.8%


## Stand and runway geometry

Runway thresholds, runway length and the taxiway network come from the X-Plane Scenery Gateway (GPL v2 or later).
Stand positions come from OpenStreetMap (ODbL), with X-Plane as fallback for stands OSM does not know; a group of
candidate positions spread over more than 100 m counts as ambiguous and stays empty. The matching lives in
`adsb_airport_geometry.py`, which `04_adsb_movement` uses as well, so both steps place a stand at the same point.

Features: straight distance from stand to threshold, its split along and across the runway, the shortest path over
the taxiway network, and the runway length.

In [16]:
sys.path.insert(0, str(ROOT / "data"))
import adsb_airport_geometry as geometry

GEOMETRY = ["stand_to_runway_distance_m", "stand_runway_parallel_m", "stand_runway_perpendicular_m",
            "taxi_route_distance_m", "runway_length_m"]
keys = geometry.geometry_keys(pd.concat([d[geometry.GEOMETRY_KEYS] for d in departures.values()]))
airports_geometry = {}
for icao in sorted(keys["ADEP_mvt"].dropna().unique()):
    airport = geometry.parse_airport_geometry(geometry.load_gateway_geometry(icao))
    requested = keys.loc[keys["ADEP_mvt"].eq(icao), "STAND_mvt"]
    airport["stands"], audit = geometry.select_stand_positions(icao, requested, geometry.load_osm_stands(icao),
                                                               airport["stand_candidates"])
    airport["stand_snaps"] = {s: geometry.nearest_geometry_node(p, airport["starts"], airport["nodes"], geometry.STAND_SNAP_M)
                              for s, p in airport["stands"].items()}
    airports_geometry[icao] = airport
    print(f"{icao}: stands placed {len(airport['stands'])} of {requested.nunique()}, "
          f"sources {audit['stand_position_source'].value_counts().to_dict()}")

EDDF recommended scenery 103368 cache: xplane_EDDF_103368.json


EDDF: stands placed 239 of 256, sources {'exact': 167, 'normalized_ref': 41, 'mean_same_ref': 20, 'missing': 16, 'parent_stand_estimate': 6, 'xplane_fallback': 3, 'mean_variants': 1, 'ambiguous_spread': 1, 'gate_estimate': 1}
EDDM recommended scenery 111124 cache: xplane_EDDM_111124.json


EDDM: stands placed 179 of 215, sources {'exact': 110, 'xplane_fallback': 45, 'missing': 36, 'mean_variants': 24}
EGLL recommended scenery 111341 cache: xplane_EGLL_111341.json


EGLL: stands placed 219 of 233, sources {'exact': 165, 'mean_same_ref': 51, 'missing': 14, 'xplane_fallback': 2, 'gate_estimate': 1}


EHAM recommended scenery 111943 cache: xplane_EHAM_111943.json


...

LEMD: stands placed 379 of 384, sources {'exact': 368, 'mean_same_ref': 9, 'missing': 5, 'xplane_fallback': 2}
LFPG recommended scenery 111022 cache: xplane_LFPG_111022.json


LFPG: stands placed 409 of 416, sources {'mean_same_ref': 216, 'exact': 155, 'xplane_fallback': 25, 'normalized_ref': 11, 'missing': 6, 'parent_stand_estimate': 2, 'ambiguous_spread': 1}
LIRF recommended scenery 110482 cache: xplane_LIRF_110482.json


LIRF: stands placed 137 of 143, sources {'exact': 92, 'mean_same_ref': 41, 'missing': 6, 'xplane_fallback': 4}
LSZH recommended scenery 106102 cache: xplane_LSZH_106102.json


LSZH: stands placed 197 of 220, sources {'exact': 184, 'missing': 23, 'mean_same_ref': 13}
LTFM recommended scenery 105504 cache: xplane_LTFM_105504.json


LTFM: stands placed 386 of 422, sources {'exact': 238, 'mean_same_ref': 146, 'ambiguous_spread': 19, 'missing': 17, 'parent_stand_estimate': 1, 'xplane_fallback': 1}


In [17]:
unique = keys.drop_duplicates().reset_index(drop=True)
values = [geometry.geometry_combination(airports_geometry[a], s, r) if a in airports_geometry
          else dict.fromkeys(geometry.GEOMETRY_FEATURES, np.nan) for a, s, r in unique.itertuples(index=False)]
lookup = pd.concat([unique, pd.DataFrame(values)[GEOMETRY]], axis=1)
for name, dep in departures.items():
    added = geometry.geometry_keys(dep).merge(lookup, on=geometry.GEOMETRY_KEYS, how="left", validate="many_to_one")
    for column in GEOMETRY:
        features[name][column] = added[column].to_numpy()
features[TRAINING["departures"]][GEOMETRY].describe().T.round(0)

,count,mean,std,min,25%,50%,75%,max
stand_to_runway_distance_m,2067183.0,2183.0,1193.0,193.0,1202.0,1974.0,2889.0,8024.0
stand_runway_parallel_m,2067183.0,834.0,1760.0,-5229.0,-159.0,911.0,1934.0,7703.0
stand_runway_perpendicular_m,2067183.0,1224.0,948.0,1.0,602.0,927.0,1453.0,6869.0
taxi_route_distance_m,2055513.0,3000.0,1456.0,262.0,1966.0,2844.0,3889.0,10142.0
runway_length_m,2084589.0,3564.0,527.0,2024.0,3193.0,3661.0,4000.0,4201.0


## ADS-B movement

From `04_adsb_movement`: the first sustained movement of the aircraft on the ground, the first ADS-B position and the
distances of the first movement point to the stand and to the runway threshold. The times are stored as absolute
timestamps and turned into seconds before take-off here, with each file's own take-off time.

In [18]:
adsb = pd.read_parquet(ADSB_FEATURES, columns=["MVT_ID_mvt", "adsb_movement_start_time", "adsb_first_observation_time",
                                                "adsb_movement_to_gate_direct_m", "adsb_movement_to_runway_direct_m"])
adsb["MVT_ID_mvt"] = adsb["MVT_ID_mvt"].astype("string")
adsb = adsb.set_index("MVT_ID_mvt")
for name, dep in departures.items():
    ids = dep["MVT_ID_mvt"].astype("Int64").astype("string")
    found = adsb.reindex(ids)
    mvt = pd.to_datetime(dep["MVT_TIME_UTC_mvt"], utc=True).to_numpy()
    features[name]["adsb_movement_to_mvt_sec"] = (mvt - pd.to_datetime(found["adsb_movement_start_time"], utc=True).to_numpy()) / np.timedelta64(1, "s")
    features[name]["adsb_first_observation_to_mvt_sec"] = (mvt - pd.to_datetime(found["adsb_first_observation_time"], utc=True).to_numpy()) / np.timedelta64(1, "s")
    features[name]["adsb_movement_to_gate_direct_m"] = found["adsb_movement_to_gate_direct_m"].to_numpy()
    features[name]["adsb_movement_to_runway_direct_m"] = found["adsb_movement_to_runway_direct_m"].to_numpy()
    print(name, "with an ADS-B movement:", f"{features[name]['adsb_movement_to_mvt_sec'].notna().mean():.1%}")

train_DEP_merged_sorted.parquet with an ADS-B movement: 82.8%


ranking.parquet with an ADS-B movement: 84.2%


final_ranking.parquet with an ADS-B movement: 83.8%


## Save

The original columns plus the features. Arrival rows keep missing feature values.

In [19]:
for name, rows in own_rows.items():
    full = pd.read_parquet(CLEAN / name)
    added = features[name].reindex(full.index)
    out = pd.concat([full, added], axis=1)
    assert len(out) == len(full) and out.loc[~full.PHASE_mvt.eq("DEP"), added.columns].isna().all().all()
    out.to_parquet(OUT / name, index=False)
    print(f"{name}: {len(out):,} rows, {added.shape[1]} features")
summary = pd.DataFrame({name: f.notna().mean().round(3) for name, f in features.items()})
summary

train_DEP_merged_sorted.parquet: 2,085,047 rows, 59 features


ranking.parquet: 689,534 rows, 59 features


final_ranking.parquet: 1,341,197 rows, 59 features


,train_DEP_merged_sorted.parquet,ranking.parquet,final_ranking.parquet
mvt_minus_aobt_sec,0.989,0.985,0.986
movement_minus_scheduled_offblock_sec,1.000,1.000,1.000
delay_sec,1.000,1.000,1.000
mvt_minus_lobt_sec,0.989,0.985,0.986
mvt_minus_eobt_sec,0.989,0.985,0.986
mvt_minus_iobt_sec,0.989,0.985,0.986
mvt_minus_arvt1_sec,0.989,0.985,0.986
mvt_minus_arvt3_sec,0.989,0.985,0.986
sched_minus_eobt_sec,0.989,0.985,0.986
sched_minus_iobt_sec,0.989,0.985,0.986
